# V38 Combined Data Preparation — Steps 05–07 + 2007 Initial Share

Run the sections in this order: **Step 05 → Step 06 → Step 07 → 2007 initial fresh-FV share**. The 2007 share section is based on the final HMM household sample created in Step 06/07, so it belongs after those outputs exist. The original input notebooks and source data are not overwritten.

**Required inputs:** V38 Step 04 trip panel; verified `food_list.xlsx`; NIQ product master; Magnet purchase extracts; 2007 and 2008–2017 NIQ purchase data; demographic event TSV. Paths are configured in each section.

**Model specification:** five emission outcomes (`fv_y`, `beverage_y`, `yogurt_y`, `cereal_y`, `snack_y`); milk fields remain audit-only. Seven annual life events are retained. Income is imputed in Step 07. The 2007 pooled fresh-FV share is saved as a starting value for the HMM initial-state probability.

## Performance update

- Required purchase columns are read from each raw TSV once into a temporary Parquet projection, reused for reconciliation and aggregation, then removed after that source year succeeds. Failed runs rebuild the projection; no stale checkpoint is silently reused.
- Staged food lines are aggregated once for all years. Smaller, date-ordered trip totals are then selected by year.
- Annual demographics and the reused 2007 lookup/extract sources are materialized once per run.
- Income interpolation uses grouped forward/backward anchors and actual calendar-year gaps rather than a Python function per household. Income rules, household-year matching, duplicate checks and monetary audits are retained.
- DuckDB keeps the existing 4GB memory budget and four threads. Use a local disk with free space for temporary projections and spills; pandas annual covariates also use RAM outside the DuckDB budget. No measured full-data speedup is claimed.
- Restart the kernel before running this updated notebook. Stop any older run first to avoid concurrent writes to the same output paths.

**Validation:** full synthetic old/new pipeline comparison passed for 65 Parquet outputs and all CSV audits (excluding runtime fields), with floating-point tolerance. Fixtures include repeated Magnet keys, late-2017 records in the 2018 source, unmatched food trips and unmatched demographic household-years. Randomized income equivalence was checked separately. Full NIQ data runtime has not been benchmarked here.


# V38 Step 05 — Recode Model Nonpurchase and Construct Trip Food Spending

This section runs first in the combined V38 preparation workflow.

**Healthy Purchase Cycle | Milk–Beverage Combined HMM | 2008–2017 | Python + DuckDB**

This notebook follows V38 Step 04. It keeps the five modeled category outcomes, converts category nonpurchase from `NULL` to `0`, and constructs all-food spending independently from raw Nielsen purchase lines and Magnet purchase lines. The combined Beverage spending from Step 04 is **never added to `food_spend`**: beverage, plain milk, and flavored milk purchases enter the denominator only once through the raw purchase-line construction (subject to the food-module list and UPC match).

Milk and original Beverage source fields remain audit fields. `milk_y` and `beverage_y_original` are not modeled outcomes and remain nullable in the corrected panel. Canned milk and buttermilk may remain part of total food spending if included by the pre-existing food-module list, even though they are excluded from the V38 Beverage classification.

Outputs are written to a separate local V38 data directory, outside GitHub. Source data and V37 outputs are not overwritten.

In [1]:
from pathlib import Path
import duckdb, pandas as pd, time, shutil

REPO = Path(r"C:/Users/rwen13/Documents/healthy_purchase_cycle")
DATA = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
RAW_HMS = Path(r"C:/Users/rwen13/Documents/2004-2020_Panel_Data/nielsen_extracts/HMS")
FOOD_LIST = REPO / "data_cleaning/upc_classification/food_list.xlsx"
MASTER = RAW_HMS / "Master_Files/Latest/products.tsv"

V38 = DATA / "intermediate/v38_milk_beverage_combined"
SOURCE_PANEL = V38 / "04_category_trip_merge/trip_purchase_panel"
CORRECTED_PANEL = V38 / "05_trip_food_spending/trip_purchase_panel_y0"
FOOD_PANEL = V38 / "05_trip_food_spending/trip_food_spending"
STAGE = V38 / "05_trip_food_spending/stage"
AUDIT = V38 / "05_trip_food_spending/validation"
TEMP = DATA / "temp/v38_milk_beverage_combined/food_spending"
MAGNET_DIR = DATA / "intermediate/magnet_food_purchases"
YEARS = range(2008, 2018)
SOURCE_YEARS = range(2008, 2019)   # 2018 source may contain late-2017 purchases
MEMORY_LIMIT, THREADS = '4GB', 4

# Confirmed all-food spending convention: net of coupon, floored at zero.
# This is computed from source purchase lines once; category spending is not added again.
USE_CAPPED_PRICE = True

for folder in [CORRECTED_PANEL, FOOD_PANEL, STAGE, AUDIT, TEMP]:
    folder.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(database=':memory:')
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory='{TEMP.as_posix()}'")

def qpath(p):
    return "'" + Path(p).as_posix().replace("'", "''") + "'"
def qname(s):
    return '"' + s.replace('"','""') + '"'
def parquet_ref(path):
    return f'read_parquet({qpath(path)})'
def raw_ref(path):
    return f"read_csv({qpath(path)}, delim='\t', header=true, all_varchar=true)"

assert FOOD_LIST.is_file(), FOOD_LIST
assert MASTER.is_file(), MASTER
print('Ready: DuckDB', duckdb.__version__)
print('V38 Step 04 input:', SOURCE_PANEL)
print('V38 Step 05 output:', V38 / '05_trip_food_spending')

Ready: DuckDB 1.5.5
V38 Step 04 input: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel
V38 Step 05 output: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\05_trip_food_spending


## A. Recode model-category nonpurchase outcomes

The V38 Step 04 trip union contains five model outcomes (`fv_y`, combined `beverage_y`, `yogurt_y`, `cereal_y`, and `snack_y`). A `NULL` means that category was not purchased on that trip. This cell writes a separate V38 file with those **five model outcomes** recoded from `NULL` to `0`.

`milk_y` and `beverage_y_original` are retained as nullable audit fields, not recoded or used as separate emissions. Milk spending components and original Beverage spending are also retained as audit fields. Only model-category spending variables are zero-filled for nonpurchase.

In [2]:
MODEL_OUTCOMES = ['fv_y','beverage_y','yogurt_y','cereal_y','snack_y']
AUDIT_OUTCOMES = ['milk_y','beverage_y_original']
MODEL_SPENDS = [
    'fresh_spending','non_fresh_spending',
    'skimmed_yogurt_spending','low_fat_yogurt_spending','other_yogurt_spending',
    'low_sugar_cereal_spending','medium_sugar_cereal_spending','high_sugar_cereal_spending',
    'healthy_snack_spending','unhealthy_snack_spending',
    'healthy_beverage_spending','unhealthy_beverage_spending'
]
AUDIT_SPENDS = [
    'plain_milk_spending','flavored_milk_spending','total_milk_spending',
    'healthy_beverage_spending_original','unhealthy_beverage_spending_original'
]
EXPECTED = {'fv_y':{1,2},'beverage_y':{1,2},'yogurt_y':{1,2},
            'cereal_y':{1,2,3},'snack_y':{1,2}}
recode_audit=[]
for year in YEARS:
    source = SOURCE_PANEL / f'panel_year={year}' / 'trips.parquet'
    assert source.is_file(), f'Run V38 Step 04 first: {source}'
    rel=parquet_ref(source)
    cols = set(con.execute(f'DESCRIBE SELECT * FROM {rel}').df()['column_name'])
    required = set(['hhid','panel_year','trip_id','purchase_date',*MODEL_OUTCOMES,*AUDIT_OUTCOMES,*MODEL_SPENDS,*AUDIT_SPENDS])
    assert required <= cols, f'Missing columns in {year}: {sorted(required - cols)}'
    checks = ', '.join(f"COUNT(*) FILTER (WHERE {qname(y)} IS NOT NULL AND {qname(y)} NOT IN ({','.join(map(str,sorted(EXPECTED[y])))})) AS {qname(y)}" for y in MODEL_OUTCOMES)
    audit_expressions = [checks, 'COUNT(*)']
    audit_expressions += [f'COUNT(*) FILTER (WHERE {qname(y)} IS NULL)' for y in MODEL_OUTCOMES]
    audit_expressions += ['COUNT(*) FILTER (WHERE milk_y IS NULL)',
                          'COUNT(*) FILTER (WHERE beverage_y_original IS NULL)']
    combined_audit = con.execute(f"SELECT {', '.join(audit_expressions)} FROM {rel}").fetchone()
    invalid = combined_audit[:len(MODEL_OUTCOMES)]
    if any(invalid):
        raise ValueError(f'{year} has unexpected non-null model outcome codes: {dict(zip(MODEL_OUTCOMES, invalid))}')
    nulls = combined_audit[len(MODEL_OUTCOMES)+1:-2]
    dst= CORRECTED_PANEL / f'panel_year={year}' / 'trips.parquet'
    dst.parent.mkdir(parents=True,exist_ok=True)
    select_expr = []
    schema = con.execute(f'DESCRIBE SELECT * FROM {rel}').df()['column_name'].tolist()
    for field in schema:
        if field in MODEL_OUTCOMES:
            select_expr.append(f'COALESCE({qname(field)}, 0)::INTEGER AS {qname(field)}')
        elif field in MODEL_SPENDS:
            select_expr.append(f'COALESCE({qname(field)}, 0)::DOUBLE AS {qname(field)}')
        else:
            # Keep Milk and source-Beverage audit outcomes/spending unchanged.
            select_expr.append(qname(field))
    con.execute(f"COPY (SELECT {', '.join(select_expr)} FROM {rel}) TO {qpath(dst)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    row={'panel_year':year,'trips':combined_audit[len(MODEL_OUTCOMES)]}
    row.update({f'{y}_null_to_zero':n for y,n in zip(MODEL_OUTCOMES,nulls)})
    row['milk_y_null_audit_rows']=combined_audit[-2]
    row['original_beverage_y_null_audit_rows']=combined_audit[-1]
    recode_audit.append(row)
pd.DataFrame(recode_audit).to_csv(AUDIT/'model_outcome_zero_recode.csv',index=False)
display(pd.DataFrame(recode_audit))

,panel_year,trips,fv_y_null_to_zero,beverage_y_null_to_zero,yogurt_y_null_to_zero,cereal_y_null_to_zero,snack_y_null_to_zero,milk_y_null_audit_rows,original_beverage_y_null_audit_rows
0,2008,3368393,1593600,1261231,3001486,2936829,1975408,2356375,1810250
1,2009,3699872,1685774,1416650,3278063,3223296,2159255,2602922,2009543
2,2010,4027911,1804812,1566500,3539490,3515224,2338537,2857650,2198881
3,2011,4206761,1840998,1683933,3685404,3699762,2417455,3023806,2332274
4,2012,4154835,1826675,1669380,3648541,3673275,2407662,3045381,2279787
5,2013,4119659,1799994,1678517,3601389,3662514,2382558,3064157,2258895
6,2014,4196848,1801094,1731969,3667676,3744989,2415932,3169217,2295050
7,2015,4145029,1718136,1711120,3620775,3712210,2404246,3150090,2246356
8,2016,4122803,1691638,1701372,3615767,3714182,2392120,3159358,2216568
9,2017,3704316,1478363,1565107,3256464,3366296,2142876,2861654,2021019


## B. Load verified food-module list and build a UPC/module lookup

The complete `food_list.xlsx` continues to define which modules enter the food denominator. UPC and UPC-version remain strings; only module codes are normalized. Conflicting master-product mappings stop the run.

This cell also audits whether plain/flavored milk modules (3625/3592) and excluded-from-beverage canned milk/buttermilk modules (1297/3626) are included in the all-food module list. Any included module contributes to `food_spend` once from source purchase lines; none of the Step 04 category spending is added to the denominator.

In [3]:
food_list=pd.read_excel(FOOD_LIST, dtype={'product_module_code':'string'})
assert 'product_module_code' in food_list.columns
food_list['module']=(food_list['product_module_code'].astype('string').str.strip()
    .str.replace(r'\.0+$','',regex=True).str.replace(r'^0+(?=\d)', '',regex=True))
modules=sorted(set(food_list['module'].dropna())-{'','<NA>'})
assert modules, 'Empty food-module list'
con.register('food_module_df',pd.DataFrame({'module':modules}))
con.execute('CREATE OR REPLACE TEMP TABLE food_modules AS SELECT DISTINCT module FROM food_module_df')
print('Allowed food modules:',len(modules))
master_cols=set(con.execute(f'DESCRIBE SELECT * FROM {raw_ref(MASTER)}').df()['column_name'])
for req in ['upc','upc_ver_uc','product_module_code']:
    assert req in master_cols, f'Master missing {req}'
con.execute(f"""
 CREATE OR REPLACE TEMP TABLE product_module_lookup AS
 SELECT TRIM(upc) AS upc, TRIM(upc_ver_uc) AS upc_ver_uc,
        CASE WHEN ltrim(regexp_replace(TRIM(product_module_code), '\\.0+$',''), '0') = ''
             THEN '0' ELSE ltrim(regexp_replace(TRIM(product_module_code), '\\.0+$',''), '0') END AS module
 FROM {raw_ref(MASTER)}
 WHERE upc IS NOT NULL AND upc_ver_uc IS NOT NULL
""")
conflicts=con.execute('''SELECT COUNT(*) FROM (
 SELECT upc,upc_ver_uc FROM product_module_lookup
 GROUP BY 1,2 HAVING COUNT(DISTINCT module)>1)''').fetchone()[0]
if conflicts:
    raise ValueError(f'Master has {conflicts} UPC-version keys with conflicting modules; resolve before aggregation')
con.execute('''CREATE OR REPLACE TEMP TABLE food_product_lookup AS
 SELECT DISTINCT p.upc,p.upc_ver_uc,p.module FROM product_module_lookup p
 INNER JOIN food_modules m ON p.module=m.module''')
print('Food UPC-versions:',con.execute('SELECT COUNT(*) FROM food_product_lookup').fetchone()[0])

# Check denominator coverage for the milk modules used by V38 plus legacy dairy
# modules. Exclusion from Beverage does not remove a product from total food spend
# when the existing food-module definition includes it.
milk_modules = ['3625','3592','1297','3626']
food_list_coverage = con.execute("""
    SELECT m.module,
           COUNT(DISTINCT p.upc || '|' || p.upc_ver_uc) AS master_upc_versions,
           COUNT(DISTINCT f.upc || '|' || f.upc_ver_uc) AS included_food_upc_versions,
           m.module IN (SELECT module FROM food_modules) AS module_in_food_list
    FROM (SELECT UNNEST(?) AS module) m
    LEFT JOIN product_module_lookup p ON p.module=m.module
    LEFT JOIN food_product_lookup f ON f.upc=p.upc AND f.upc_ver_uc=p.upc_ver_uc
    GROUP BY m.module ORDER BY m.module
""", [milk_modules]).df()
food_list_coverage.to_csv(AUDIT/'milk_module_food_denominator_coverage.csv',index=False)
display(food_list_coverage)

Allowed food modules: 891
Food UPC-versions: 2355125


,module,master_upc_versions,included_food_upc_versions,module_in_food_list
0,1297,810,810,True
1,3592,4352,4352,True
2,3625,14041,14041,True
3,3626,1146,1146,True


## C. Stage raw food spending, one source year at a time

Regular food is identified by the verified `food_list.xlsx` module list and original purchase lines. Magnet food comes from the existing extracted Magnet purchase files. Repeated trip/UPC/version keys are retained. Before excluding Magnet keys from the regular-food branch, this notebook verifies that Magnet and original purchase-line multiplicities match for each key.

Milk is **not joined to or added from the Step 04 category panel** here. If modules 3625/3592 occur in `food_list.xlsx`, their source purchase lines enter the all-food denominator through the regular-food branch exactly once. The staging files retain module codes so the notebook can separately audit milk-module dollars already inside `food_spend`. Canned milk and buttermilk remain part of total food expenditure whenever their modules are part of the existing food list.

In [4]:
# Run this inexpensive schema/preflight audit before launching the yearly scans.
for sy in SOURCE_YEARS:
    base=RAW_HMS/str(sy)/'Annual_Files'
    for label,name in [('purchases',f'purchases_{sy}.tsv'),('trips',f'trips_{sy}.tsv')]:
        assert (base/name).is_file(), f'Missing {label}: {base/name}'
    magnet=MAGNET_DIR/f'magnet_food_purchases_{sy}.tsv'
    assert magnet.is_file(), f'Missing extracted magnet file: {magnet}'
print('All source-year purchase, trip and Magnet files found.')
print('V38 trip panel directory:', SOURCE_PANEL)
print('Food module list remains unchanged; inspect milk module coverage audit before interpreting denominator coverage.')
for label, path, needed in [
 ('purchases',RAW_HMS/'2008/Annual_Files/purchases_2008.tsv', ['upc','upc_ver_uc','trip_code_uc','total_price_paid','coupon_value']),
 ('trips',RAW_HMS/'2008/Annual_Files/trips_2008.tsv',['trip_code_uc','household_code','purchase_date']),
 ('magnet',MAGNET_DIR/'magnet_food_purchases_2008.tsv',['upc','upc_ver_uc','trip_code_uc','magnet_food_spend_increment_capped','purchase_date'])]:
    cols=set(con.execute(f'DESCRIBE SELECT * FROM {raw_ref(path)}').df()['column_name'])
    assert set(needed)<=cols, f'{label}: missing {set(needed)-cols}'
    print(label,'columns OK')

All source-year purchase, trip and Magnet files found.
V38 trip panel directory: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel
Food module list remains unchanged; inspect milk module coverage audit before interpreting denominator coverage.
purchases columns OK
trips columns OK
magnet columns OK


In [5]:
# Each source-year file is processed separately; DuckDB executes file scans on disk.
# Intermediate Parquet contains only the records needed for analysis years.
source_audit=[]
for sy in SOURCE_YEARS:
    started=time.perf_counter()
    print(f'Starting source year {sy}', flush=True)
    base=RAW_HMS/str(sy)/'Annual_Files'
    p=raw_ref(base/f'purchases_{sy}.tsv')
    t=raw_ref(base/f'trips_{sy}.tsv')
    m=raw_ref(MAGNET_DIR/f'magnet_food_purchases_{sy}.tsv')
    stage_dir=STAGE/f'source_year={sy}'
    stage_dir.mkdir(parents=True,exist_ok=True)
    # Treat 2018 source-year boundary dates like all other purchase dates.
    con.execute(f"""CREATE OR REPLACE TEMP TABLE year_trips AS
        SELECT TRIM(trip_code_uc) AS trip_code_uc,
               TRIM(household_code) AS hhid,
               TRY_CAST(purchase_date AS DATE) AS purchase_date
        FROM {t}""")
    dup_trip=con.execute('SELECT COUNT(*) FROM (SELECT trip_code_uc FROM year_trips GROUP BY 1 HAVING COUNT(*)>1)').fetchone()[0]
    if dup_trip: raise ValueError(f'{sy}: {dup_trip} duplicate trip_code_uc in trips source')
    # Materialize Magnet keys and amounts for a relatively small source-year output.
    con.execute(f"""CREATE OR REPLACE TEMP TABLE year_magnet AS
        SELECT TRIM(trip_code_uc) trip_code_uc, TRIM(upc) upc,
               TRIM(upc_ver_uc) upc_ver_uc,
               TRY_CAST(purchase_date AS DATE) purchase_date,
               TRIM(household_code) hhid,
               TRY_CAST(magnet_food_spend_increment_capped AS DOUBLE) amount
        FROM {m}""")
    # Repeated trip/UPC/version combinations are valid purchase lines.
    # Never deduplicate them: amounts from every row are required.
    # Compare purchase-line multiplicities against the raw source instead.
    con.execute("""CREATE OR REPLACE TEMP TABLE year_magnet_keys AS
      SELECT trip_code_uc, upc, upc_ver_uc, COUNT(*) AS magnet_rows
      FROM year_magnet
      GROUP BY 1,2,3""")
    magnet_dups=con.execute("""SELECT COUNT(*) FROM year_magnet_keys
      WHERE magnet_rows > 1""").fetchone()[0]

    # Read original purchase lines with original UPC keys preserved as strings.
    # Rebuild this narrow on-disk projection on every run; never trust an old cache.
    purchase_cache = stage_dir / '_purchase_projection.parquet'
    print(f'{sy}: reading purchases TSV once into a compact projection...', flush=True)
    con.execute(f"""COPY (
      SELECT TRIM(trip_code_uc) trip_code_uc, TRIM(upc) upc,
             TRIM(upc_ver_uc) upc_ver_uc,
             TRY_CAST(total_price_paid AS DOUBLE) total_price_paid,
             COALESCE(TRY_CAST(coupon_value AS DOUBLE),0) coupon_value
      FROM {p}
    ) TO {qpath(purchase_cache)} (FORMAT PARQUET, COMPRESSION SNAPPY)""")
    con.execute(f'CREATE OR REPLACE TEMP VIEW year_purchases AS SELECT * FROM {parquet_ref(purchase_cache)}')

    # SEMI JOIN prevents repeated Magnet keys from multiplying purchase rows.
    # A key-level anti-join is valid only if every Magnet purchase line
    # corresponds to exactly one raw purchase line with the same key.
    con.execute("""CREATE OR REPLACE TEMP TABLE year_raw_magnet_key_counts AS
      SELECT p.trip_code_uc, p.upc, p.upc_ver_uc,
             COUNT(*) AS purchase_rows
      FROM year_purchases p
      SEMI JOIN year_magnet_keys m
        USING (trip_code_uc, upc, upc_ver_uc)
      GROUP BY 1,2,3""")
    overlap=con.execute("""SELECT
        COUNT(*) FILTER (WHERE p.purchase_rows IS NULL) AS unmatched_keys,
        COUNT(*) FILTER (WHERE p.purchase_rows IS NOT NULL
                        AND p.purchase_rows <> m.magnet_rows) AS unequal_keys,
        COALESCE(SUM(m.magnet_rows),0) AS magnet_rows,
        COALESCE(SUM(p.purchase_rows),0) AS matched_raw_rows
      FROM year_magnet_keys m LEFT JOIN year_raw_magnet_key_counts p
        USING (trip_code_uc, upc, upc_ver_uc)""").fetchone()
    unmatched_keys, unequal_keys, magnet_rows, raw_rows = overlap
    print(f'{sy}: repeated Magnet trip/UPC keys={magnet_dups:,}; '
          f'unmatched={unmatched_keys:,}; unequal multiplicities={unequal_keys:,}')
    if unmatched_keys or unequal_keys:
        samples=con.execute("""SELECT m.trip_code_uc,m.upc,m.upc_ver_uc,
                  m.magnet_rows,p.purchase_rows
            FROM year_magnet_keys m LEFT JOIN year_raw_magnet_key_counts p
              USING (trip_code_uc,upc,upc_ver_uc)
            WHERE p.purchase_rows IS NULL OR m.magnet_rows <> p.purchase_rows
            LIMIT 15""").df()
        display(samples)
        raise ValueError(f'{sy}: Magnet/raw purchase-line counts mismatch '
                         f'(unmatched={unmatched_keys}, unequal={unequal_keys}). '
                         'Cannot safely apply key-level anti-join.')
    # All same-key raw rows are accounted for in Magnet; exclude that
    # key from regular foods and add all Magnet increments exactly once.
    ambig = unequal_keys
    # Matched regular-food purchases, excluding corresponding Magnet keys.
    out_regular=stage_dir/'regular.parquet'
    con.execute(f"""COPY (
      SELECT tr.hhid, tr.purchase_date, tr.trip_code_uc, f.module AS product_module_code,
             CAST(GREATEST(COALESCE(p.total_price_paid - p.coupon_value,0),0) AS DOUBLE) AS food_amount
      FROM year_purchases p
      INNER JOIN food_product_lookup f USING (upc, upc_ver_uc)
      INNER JOIN year_trips tr USING (trip_code_uc)
      WHERE tr.hhid IS NOT NULL AND tr.purchase_date BETWEEN DATE '2008-01-01' AND DATE '2017-12-31'
        AND NOT EXISTS (
          SELECT 1 FROM year_magnet_keys m
          WHERE m.trip_code_uc=p.trip_code_uc AND m.upc=p.upc AND m.upc_ver_uc=p.upc_ver_uc
        )
    ) TO {qpath(out_regular)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    out_magnet=stage_dir/'magnet.parquet'
    con.execute(f"""COPY (
      SELECT hhid, purchase_date, trip_code_uc,
             CAST(NULL AS VARCHAR) AS product_module_code,
             CAST(COALESCE(amount,0) AS DOUBLE) AS food_amount
      FROM year_magnet
      WHERE hhid IS NOT NULL AND purchase_date BETWEEN DATE '2008-01-01' AND DATE '2017-12-31'
    ) TO {qpath(out_magnet)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    regular_n=con.execute(f'SELECT COUNT(*) FROM {parquet_ref(out_regular)}').fetchone()[0]
    magnet_n=con.execute(f'SELECT COUNT(*) FROM {parquet_ref(out_magnet)}').fetchone()[0]
    source_audit.append({'source_year':sy,'regular_food_purchase_rows':regular_n,
        'magnet_food_purchase_rows':magnet_n,'duplicate_magnet_keys':magnet_dups,
        'unmatched_magnet_keys':unmatched_keys,
        'unequal_row_count_keys':unequal_keys,'magnet_rows_reconciled':magnet_rows,
        'raw_rows_reconciled':raw_rows,'seconds':round(time.perf_counter()-started,1)})
    print(f'{sy}: regular={regular_n:,}, magnet={magnet_n:,}', flush=True)
    con.execute('DROP VIEW year_purchases')
    purchase_cache.unlink()  # Keep only one source year's projection on disk.
    for table in ('year_trips', 'year_magnet', 'year_magnet_keys', 'year_raw_magnet_key_counts'):
        con.execute(f'DROP TABLE IF EXISTS {table}')
pd.DataFrame(source_audit).to_csv(AUDIT/'food_spending_source_audit.csv',index=False)
display(pd.DataFrame(source_audit))

Starting source year 2008
2008: reading purchases TSV once into a compact projection...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2008: repeated Magnet trip/UPC keys=874,677; unmatched=0; unequal multiplicities=0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2008: regular=46,438,262, magnet=4,480,556
Starting source year 2009
2009: reading purchases TSV once into a compact projection...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (36 outputs left out)

2018: repeated Magnet trip/UPC keys=239,923; unmatched=0; unequal multiplicities=0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2018: regular=169,526, magnet=12,690


,source_year,regular_food_purchase_rows,magnet_food_purchase_rows,duplicate_magnet_keys,unmatched_magnet_keys,unequal_row_count_keys,magnet_rows_reconciled,raw_rows_reconciled,seconds
0,2008,46438262,4480556,874677,0,0,4510411,4510411,36.4
1,2009,46073476,4924316,953768,0,0,4924316,4924316,39.8
2,2010,44912911,5199900,1003710,0,0,5199900,5199900,42.8
3,2011,47129997,5809676,1114275,0,0,5809676,5809676,51.8
4,2012,47262733,3065814,237045,0,0,3065814,3065814,55.3
5,2013,47489893,3111419,219178,0,0,3111419,3111419,57.5
6,2014,48715836,3442975,226244,0,0,3442975,3442975,58.3
7,2015,47950119,3701490,223717,0,0,3701490,3701490,58.3
8,2016,50885277,4067281,244830,0,0,4067281,4067281,64.2
9,2017,49880310,4197064,247678,0,0,4197064,4197064,65.0


## D. Sum food dollars by actual trip; match against the V38 corrected panel

The corrected V38 Step 04 panel is read from the separate V38 directory. Food totals are grouped before joining, so purchase lines cannot multiply trip rows. The `food_spend` value comes only from original regular-food purchase lines and Magnet increments. It does **not** add any category-level spending fields from Step 04.

The notebook reports match rates rather than converting unmatched trips to zero. `milk_module_food_spend` is an audit component already contained within `regular_food_spend` and `food_spend`; it is not added again. Review its coverage with `milk_module_food_denominator_coverage.csv`.

In [6]:
# Build per-year totals from staged source-year files using DuckDB wildcard paths.
# Group before joining: never join purchase rows directly to the trip panel.
regular_files=[STAGE/f'source_year={sy}'/'regular.parquet' for sy in SOURCE_YEARS]
magnet_files=[STAGE/f'source_year={sy}'/'magnet.parquet' for sy in SOURCE_YEARS]
def rel_for(paths):
    return 'read_parquet(['+', '.join(qpath(p) for p in paths)+'])'
regular_rel=rel_for(regular_files)
magnet_rel=rel_for(magnet_files)
# One aggregation across all source years, preserving cross-source trip overlaps.
# Only purchase dates in 2008–2017 were staged. Ordered Parquet permits date-range pruning.
all_food_totals = STAGE / 'food_trip_totals_all_years.parquet'
print('Aggregating staged food lines once for all calendar years...', flush=True)
con.execute(f"""COPY (
    SELECT hhid, purchase_date, trip_code_uc,
           SUM(regular_amount) AS regular_food_spend,
           SUM(magnet_amount) AS magnet_food_spend,
           SUM(regular_amount+magnet_amount) AS food_spend,
           SUM(CASE WHEN product_module_code IN ('3625','3592','1297','3626')
                    THEN regular_amount ELSE 0 END) AS milk_module_food_spend
    FROM (
      SELECT hhid,purchase_date,trip_code_uc,food_amount AS regular_amount,0::DOUBLE AS magnet_amount,product_module_code
      FROM {regular_rel}
      UNION ALL
      SELECT hhid,purchase_date,trip_code_uc,0::DOUBLE,food_amount,CAST(NULL AS VARCHAR)
      FROM {magnet_rel}
    ) z GROUP BY 1,2,3 ORDER BY purchase_date
) TO {qpath(all_food_totals)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
food_audit=[]
for year in YEARS:
    t0=time.perf_counter()
    panel=CORRECTED_PANEL/f'panel_year={year}'/'trips.parquet'
    assert panel.is_file()
    # Stage annual totals so the final join is small and deterministic.
    food_totals=STAGE/f'food_trip_totals_{year}.parquet'
    con.execute(f"""COPY (
      SELECT * FROM {parquet_ref(all_food_totals)}
      WHERE purchase_date >= DATE '{year}-01-01' AND purchase_date < DATE '{year+1}-01-01'
    ) TO {qpath(food_totals)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    f=parquet_ref(food_totals); c=parquet_ref(panel)
    dup=con.execute(f'''SELECT COUNT(*) FROM (SELECT hhid,purchase_date,trip_code_uc FROM {f}
      GROUP BY ALL HAVING COUNT(*)>1)''').fetchone()[0]
    assert dup==0, f'Duplicate food trip keys for {year}'
    dest=FOOD_PANEL/f'panel_year={year}'/'trips.parquet'
    dest.parent.mkdir(parents=True,exist_ok=True)
    con.execute(f"""COPY (
      SELECT c.hhid, c.panel_year, c.trip_id, c.purchase_date,
             f.regular_food_spend, f.magnet_food_spend, f.food_spend,
             f.milk_module_food_spend,
             f.trip_code_uc IS NOT NULL AS food_trip_matched
      FROM {c} c LEFT JOIN {f} f
        ON c.hhid=f.hhid AND c.purchase_date=f.purchase_date
           AND c.trip_id=f.trip_code_uc
    ) TO {qpath(dest)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    o=parquet_ref(dest)
    counts=con.execute(f'''SELECT COUNT(*) AS trips, COUNT(*) FILTER (WHERE food_trip_matched) AS matched,
      COUNT(*) FILTER (WHERE NOT food_trip_matched) AS unmatched,
      SUM(food_spend) AS food_dollars,
      COUNT(*) FILTER (WHERE food_spend < 0) AS negative_spend
      FROM {o}''').fetchone()
    row=dict(zip(['panel_year','matched','unmatched','food_dollars','negative_spend'],[year,*counts[1:]]))
    row['trips']=counts[0]; row['match_rate']=counts[1]/counts[0] if counts[0] else None
    row['matched_milk_module_food_dollars']=con.execute(f'SELECT SUM(milk_module_food_spend) FROM {o}').fetchone()[0]
    row['seconds']=round(time.perf_counter()-t0,1)
    food_audit.append(row)
    print(f'{year}: {counts[1]:,}/{counts[0]:,} category trips matched to food source; rate {row["match_rate"]:.1%}')
pd.DataFrame(food_audit).to_csv(AUDIT/'trip_food_spending_merge_audit.csv',index=False)
display(pd.DataFrame(food_audit))

Aggregating staged food lines once for all calendar years...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: could not allocate block of size 4.0 KiB (3.7 GiB/3.7 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [7]:
# Reuse existing staged Parquet files; aggregate one calendar month at a time.
from pathlib import Path
from collections import defaultdict
import tempfile
import shutil
import time
import duckdb
import pandas as pd

regular_files = [
    STAGE / f"source_year={sy}" / "regular.parquet"
    for sy in SOURCE_YEARS
]
magnet_files = [
    STAGE / f"source_year={sy}" / "magnet.parquet"
    for sy in SOURCE_YEARS
]

for path in regular_files + magnet_files:
    if not path.is_file():
        raise FileNotFoundError(f"Missing staged file: {path}")


def qpath(p):
    return "'" + Path(p).as_posix().replace("'", "''") + "'"


def parquet_ref(p):
    return f"read_parquet({qpath(p)}, hive_partitioning=false)"


def rel_for(paths):
    return (
        "read_parquet(["
        + ",".join(qpath(p) for p in paths)
        + "], hive_partitioning=false)"
    )


# Release the failed query and the old connection's temporary tables.
con.close()

work_dir = Path(tempfile.mkdtemp(prefix="_food_monthly_", dir=STAGE))
spill_dir = work_dir / "spill"
spill_dir.mkdir()

con = duckdb.connect(":memory:")
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=false")
con.execute(f"SET temp_directory={qpath(spill_dir)}")
con.execute("SET partitioned_write_max_open_files=4")

# -----------------------------------------------------------
# 1. Split each staged file once by actual purchase year/month.
#    No aggregation or global sorting at this stage.
# -----------------------------------------------------------
buckets = defaultdict(list)

for kind, paths in [("regular", regular_files), ("magnet", magnet_files)]:
    for sy, path in zip(SOURCE_YEARS, paths):
        dest = work_dir / f"{kind}_{sy}"

        regular = "food_amount" if kind == "regular" else "0::DOUBLE"
        magnet = "food_amount" if kind == "magnet" else "0::DOUBLE"
        module = (
            "product_module_code" if kind == "regular" else "NULL::VARCHAR"
        )

        con.execute(f"""
            COPY (
                SELECT
                    hhid,
                    purchase_date,
                    trip_code_uc,
                    {regular} AS regular_amount,
                    {magnet} AS magnet_amount,
                    {module} AS product_module_code,
                    YEAR(purchase_date)::INTEGER AS _yr,
                    MONTH(purchase_date)::INTEGER AS _mo
                FROM {parquet_ref(path)}
            )
            TO {qpath(dest)} (
                FORMAT PARQUET,
                COMPRESSION SNAPPY,
                PARTITION_BY (_yr, _mo),
                ROW_GROUP_SIZE 16384
            )
        """)

        # Collect this month's files across ALL source years.
        for folder in dest.glob("_yr=*/_mo=*"):
            year = int(folder.parent.name.split("=", 1)[1])
            month = int(folder.name.split("=", 1)[1])

            if year not in YEARS or not 1 <= month <= 12:
                raise ValueError(f"Unexpected staged purchase date: {folder}")

            buckets[year, month].extend(sorted(folder.glob("*.parquet")))

        print(f"Partitioned {kind} source {sy}", flush=True)

# -----------------------------------------------------------
# 2. Aggregate each month, concatenate annual totals, then join.
# -----------------------------------------------------------
food_audit = []

for year in YEARS:
    started = time.perf_counter()
    panel = CORRECTED_PANEL / f"panel_year={year}" / "trips.parquet"

    if not panel.is_file():
        raise FileNotFoundError(panel)

    monthly_totals = []

    for month in range(1, 13):
        paths = buckets.get((year, month), [])

        # Preserve output schema even when a month has no food purchases.
        source = rel_for(paths) if paths else """
            (
                SELECT
                    NULL::VARCHAR AS hhid,
                    NULL::DATE AS purchase_date,
                    NULL::VARCHAR AS trip_code_uc,
                    0::DOUBLE AS regular_amount,
                    0::DOUBLE AS magnet_amount,
                    NULL::VARCHAR AS product_module_code
                WHERE FALSE
            )
        """

        target = work_dir / f"totals_{year}_{month:02d}.parquet"

        con.execute(f"""
            COPY (
                SELECT
                    hhid,
                    purchase_date,
                    trip_code_uc,
                    SUM(regular_amount) AS regular_food_spend,
                    SUM(magnet_amount) AS magnet_food_spend,
                    SUM(regular_amount + magnet_amount) AS food_spend,
                    SUM(
                        CASE
                            WHEN product_module_code IN
                                 ('3625', '3592', '1297', '3626')
                            THEN regular_amount
                            ELSE 0
                        END
                    ) AS milk_module_food_spend
                FROM {source}
                GROUP BY hhid, purchase_date, trip_code_uc
            )
            TO {qpath(target)} (
                FORMAT PARQUET,
                COMPRESSION ZSTD
            )
        """)

        duplicate = con.execute(f"""
            SELECT COUNT(*) FROM (
                SELECT hhid, purchase_date, trip_code_uc
                FROM {parquet_ref(target)}
                GROUP BY ALL
                HAVING COUNT(*) > 1
            )
        """).fetchone()[0]

        assert duplicate == 0, (
            year, month, "duplicate food trip keys"
        )

        monthly_totals.append(target)

        # Delete only this run's temporary monthly purchase partitions.
        # Original staged source-year Parquet files remain available.
        for path in paths:
            path.unlink()

        print(f"Aggregated {year}-{month:02d}", flush=True)

    # purchase_date is part of the trip key, so distinct months cannot
    # share a full trip key. Concatenation requires no further GROUP BY.
    food_totals = STAGE / f"food_trip_totals_{year}.parquet"
    pending_totals = food_totals.with_suffix(".pending.parquet")

    con.execute(f"""
        COPY (
            SELECT * FROM {rel_for(monthly_totals)}
        )
        TO {qpath(pending_totals)} (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)
    pending_totals.replace(food_totals)

    dest = FOOD_PANEL / f"panel_year={year}" / "trips.parquet"
    dest.parent.mkdir(parents=True, exist_ok=True)
    pending = dest.with_suffix(".pending.parquet")

    # Retain every category-panel trip.
    # Unmatched food spending stays NULL.
    con.execute(f"""
        COPY (
            SELECT
                c.hhid,
                c.panel_year,
                c.trip_id,
                c.purchase_date,
                f.regular_food_spend,
                f.magnet_food_spend,
                f.food_spend,
                f.milk_module_food_spend,
                f.trip_code_uc IS NOT NULL AS food_trip_matched
            FROM {parquet_ref(panel)} c
            LEFT JOIN {parquet_ref(food_totals)} f
              ON c.hhid = f.hhid
             AND c.purchase_date = f.purchase_date
             AND c.trip_id = f.trip_code_uc
        )
        TO {qpath(pending)} (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
    """)

    counts = con.execute(f"""
        SELECT
            COUNT(*),
            COUNT(*) FILTER (WHERE food_trip_matched),
            COUNT(*) FILTER (WHERE NOT food_trip_matched),
            SUM(food_spend),
            COUNT(*) FILTER (WHERE food_spend < 0),
            SUM(milk_module_food_spend)
        FROM {parquet_ref(pending)}
    """).fetchone()

    input_count = con.execute(
        f"SELECT COUNT(*) FROM {parquet_ref(panel)}"
    ).fetchone()[0]

    assert counts[0] == input_count, (
        f"{year}: trip count changed after join"
    )
    assert counts[4] == 0, f"{year}: negative food spending"

    # Replace the annual output only after the checks pass.
    pending.replace(dest)

    rate = counts[1] / counts[0] if counts[0] else None

    food_audit.append({
        "panel_year": year,
        "matched": counts[1],
        "unmatched": counts[2],
        "food_dollars": counts[3],
        "negative_spend": counts[4],
        "trips": counts[0],
        "match_rate": rate,
        "matched_milk_module_food_dollars": counts[5],
        "seconds": round(time.perf_counter() - started, 1),
    })

    pd.DataFrame(food_audit).to_csv(
        AUDIT / "trip_food_spending_merge_audit.csv",
        index=False,
    )

    print(
        f"{year}: {counts[1]:,}/{counts[0]:,} trips matched",
        flush=True,
    )

# Keep the connection usable for the following notebook cells.
con.execute(f"SET temp_directory={qpath(TEMP)}")
shutil.rmtree(work_dir)

display(pd.DataFrame(food_audit))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Partitioned regular source 2008


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Partitioned regular source 2009


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Partitioned regular source 2010


... (229 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2017: 3,704,316/3,704,316 trips matched


,panel_year,matched,unmatched,food_dollars,negative_spend,trips,match_rate,matched_milk_module_food_dollars,seconds
0,2008,3368393,0,9.461650e+07,0,3368393,1.0,4.041286e+06,24.1
1,2009,3699872,0,1.064976e+08,0,3699872,1.0,3.652871e+06,28.2
2,2010,4027911,0,1.172186e+08,0,4027911,1.0,4.046060e+06,35.1
3,2011,4206761,0,1.290157e+08,0,4206761,1.0,4.492921e+06,38.3
4,2012,4154835,0,1.292963e+08,0,4154835,1.0,4.239071e+06,39.3
5,2013,4119659,0,1.310957e+08,0,4119659,1.0,4.355882e+06,39.9
6,2014,4196848,0,1.381371e+08,0,4196848,1.0,4.224831e+06,43.5
7,2015,4145029,0,1.406503e+08,0,4145029,1.0,3.761232e+06,42.3
8,2016,4122803,0,1.387429e+08,0,4122803,1.0,3.431102e+06,46.4
9,2017,3704316,0,1.246044e+08,0,3704316,1.0,2.931987e+06,45.8


## E. Final safety checks and interpretation

- **Match coverage matters.** Inspect `trip_food_spending_merge_audit.csv` and compare V38 rates against prior validated outputs before Step 06.
- `food_spend` is independent of the five category outcomes. Combined Beverage category spending is not an input to the denominator.
- Food expenditures are counted once from source purchase lines: ordinary food purchases use `max(total_price_paid - coupon_value, 0)`; Magnet increments are added after line-multiplicity reconciliation.
- `milk_module_food_spend` is a subset of `regular_food_spend` and `food_spend`, reported only for audit. Do not add it to `food_spend` again.
- Whether each milk module is included by the unchanged complete food-module list is recorded in `milk_module_food_denominator_coverage.csv`. Canned milk/buttermilk may remain in all-food spending even though they are excluded from the V38 beverage classification.
- This construction targets trips observed in the V38 six-input union (now five modeled outcomes). The separate 2007 fresh-FV initialization is not covered.
- The next section combines the recoded V38 category panel and separate trip food-spending table into the HMM master panel using `(hhid, panel_year, trip_id, purchase_date)`.
- All restricted inputs, staging files, audits, and outputs remain in local `healthy_purchase_data`, not GitHub.

In [8]:
# Quick QA: model outcomes and food-spending values
for year in YEARS:
    food=parquet_ref(FOOD_PANEL/f'panel_year={year}'/'trips.parquet')
    panel=parquet_ref(CORRECTED_PANEL/f'panel_year={year}'/'trips.parquet')
    bad=con.execute(f'''SELECT COUNT(*) FROM {food}
        WHERE food_spend<0 OR regular_food_spend<0 OR magnet_food_spend<0
           OR milk_module_food_spend<0''').fetchone()[0]
    assert bad==0, (year,'negative spending',bad)
    missing=con.execute('SELECT '+', '.join(f'COUNT(*) FILTER (WHERE {qname(y)} IS NULL)' for y in MODEL_OUTCOMES)+f' FROM {panel}').fetchone()
    assert not any(missing), (year,'unexpected missing model outcome',missing)
    # Preserve audit NULLs and confirm modeled outcomes use only allowed codes.
    invalid=con.execute(f'''SELECT COUNT(*) FROM {panel} WHERE
        fv_y NOT IN (0,1,2) OR beverage_y NOT IN (0,1,2) OR
        yogurt_y NOT IN (0,1,2) OR cereal_y NOT IN (0,1,2,3) OR
        snack_y NOT IN (0,1,2)''').fetchone()[0]
    assert invalid==0, (year,'invalid recoded model outcome',invalid)
print('Model outcomes and food-spending checks passed. Review match-rate and milk-module coverage audits.')

Model outcomes and food-spending checks passed. Review match-rate and milk-module coverage audits.



# V38 Step 06 — Build HMM-Ready Master Panel

This section reads the Step 05 outputs from the V38 directory and writes a separate HMM master output under `06_hmm_master`. Its category and spending schemas are aligned to the five modeled V38 outcomes.


In [9]:
# Cell 1 — Packages, paths and safe DuckDB settings
from pathlib import Path
import time
import duckdb
import pandas as pd
from IPython.display import display

DATA = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
BASE = DATA / "intermediate" / "v38_milk_beverage_combined" / "05_trip_food_spending"
DEMOGRAPHICS = DATA / "processed" / "demographics" / "household_demographics_with_events_2008_2017.tsv"
CATEGORY_PANEL = BASE / "trip_purchase_panel_y0"
FOOD_PANEL = BASE / "trip_food_spending"
OUT = DATA / "intermediate" / "v38_milk_beverage_combined" / "06_hmm_master"
TRIP_OUT = OUT / "hmm_trips"
ANNUAL_OUT = OUT / "hmm_annual_covariates.parquet"
AUDIT = OUT / "validation"
TEMP = DATA / "temp" / "v38_milk_beverage_combined" / "duckdb_hmm_master"
YEARS = range(2008, 2018)
MEMORY_LIMIT = "4GB"
THREADS = 4
# Require an explicit review of mismatched food records before model estimation.
MIN_FOOD_MATCH_RATE = 0.95  # Warning threshold only; never fills missing food amounts.

for path in (TRIP_OUT, AUDIT, TEMP):
    path.mkdir(parents=True, exist_ok=True)
assert DEMOGRAPHICS.is_file(), f"Missing Step 03 demographic output: {DEMOGRAPHICS}"
for year in YEARS:
    for label, folder in (("corrected purchases", CATEGORY_PANEL), ("food spending", FOOD_PANEL)):
        f = folder / f"panel_year={year}" / "trips.parquet"
        assert f.is_file(), f"Missing {label} for {year}: {f}"

def lit(path):
    return "'" + Path(path).as_posix().replace("'", "''") + "'"
def name(s):
    return '"' + s.replace('"', '""') + '"'
def pq(path):
    return f"read_parquet({lit(path)})"

if 'con' in globals():
    con.close()
con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory={lit(TEMP)}")
print("DuckDB:", duckdb.__version__)
print("Annual source:", DEMOGRAPHICS)


DuckDB: 1.5.5
Annual source: C:\Users\rwen13\Documents\healthy_purchase_data\processed\demographics\household_demographics_with_events_2008_2017.tsv


## A — Validate annual demographics before building anything

Use the **seven** final life-event names (including `getting_divorce_or_separation`) from demographic Step 03. Keep the output compact: only model covariates and a transition-validity flag. The full, cleaned demographics remain in their original restricted TSV.


In [10]:
# Cell 2 — Annual source schema and duplicate/key checks
EVENTS = [
    "event_new_child", "getting_married", "getting_divorce_or_separation",
    "moving", "getting_job", "retirement", "other_unemployment",
]
CONTROLS = ["K_1", "income10k", "family_size_numeric"]
EXTRA = ["valid_transition"]  # Included only if present in Step 03.
KEYS = ["hhid", "panel_year", "trip_id", "purchase_date"]
OUTCOMES = ["fv_y", "beverage_y", "yogurt_y", "cereal_y", "snack_y"]
SPENDS = [
    "fresh_spending", "non_fresh_spending",
    "skimmed_yogurt_spending", "low_fat_yogurt_spending", "other_yogurt_spending",
    "low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending",
    "healthy_snack_spending", "unhealthy_snack_spending",
    "healthy_beverage_spending", "unhealthy_beverage_spending",
]
FOOD_FIELDS = ["regular_food_spend", "magnet_food_spend", "food_spend", "food_trip_matched"]

# IDs are explicitly read as VARCHAR; NULL is preserved for event/controls.
con.execute(f"""CREATE OR REPLACE TEMP TABLE demo_raw AS
    SELECT * FROM read_csv({lit(DEMOGRAPHICS)}, delim='\t', header=true, all_varchar=true)
""")
demo_cols = set(con.execute('DESCRIBE SELECT * FROM demo_raw').df().column_name)
required = {"hhid", "panel_year", *EVENTS, *CONTROLS}
assert required <= demo_cols, f"Missing demographic fields: {sorted(required - demo_cols)}"
optional = [x for x in EXTRA if x in demo_cols]
con.execute("""CREATE OR REPLACE VIEW demo_keys AS
    SELECT TRIM(hhid) AS hhid, TRY_CAST(panel_year AS INTEGER) AS panel_year
    FROM demo_raw""")
issues = con.execute("""SELECT
    COUNT(*) AS records,
    COUNT(*) FILTER (WHERE hhid IS NULL OR hhid='' OR panel_year IS NULL OR panel_year NOT BETWEEN 2008 AND 2017) AS bad_keys,
    COUNT(*) - COUNT(DISTINCT (hhid, panel_year)) AS duplicate_hh_years
    FROM demo_keys""").df()
display(issues)
assert issues.bad_keys.iloc[0] == 0, "Invalid demographic keys/years"
assert issues.duplicate_hh_years.iloc[0] == 0, "Nonunique demographic household-years"
# Numeric and event code validation. Never coerce malformed input to an event.
for field in EVENTS + ["K_1"]:
    bad = con.execute(f"""SELECT COUNT(*) FROM demo_raw
        WHERE {name(field)} IS NOT NULL AND TRIM({name(field)}) <> ''
          AND TRY_CAST({name(field)} AS INTEGER) NOT IN (0,1)
    """).fetchone()[0]
    assert bad == 0, f"Invalid {field} codes: {bad}"
print('Annual source validated.')


,records,bad_keys,duplicate_hh_years
0,499802,0,0


Annual source validated.


## B — Build annual covariates for households represented in the purchase trips

Create a distinct household-year index from all corrected purchase partitions and **join annual demographics one-to-one**. Retain a match indicator to audit excluded/no-demographic household-years. Only matched households enter the HMM outputs. Never interpret absent life-event information as zero.


In [11]:
# Cell 3 — Small household-year index and model-ready annual covariates
trip_paths = [CATEGORY_PANEL / f"panel_year={year}" / "trips.parquet" for year in YEARS]
trip_rel = 'read_parquet([' + ','.join(lit(x) for x in trip_paths) + '])'
con.execute(f"""CREATE OR REPLACE TEMP TABLE observed_hh_years AS
    SELECT DISTINCT CAST(hhid AS VARCHAR) AS hhid, CAST(panel_year AS INTEGER) AS panel_year
    FROM {trip_rel}
""")
# Check annual coverage independently of trip-amount matching.
coverage = con.execute("""SELECT o.panel_year,
    COUNT(*) AS purchase_household_years,
    COUNT(*) FILTER (WHERE d.hhid IS NOT NULL) AS matched_demographic_years,
    COUNT(*) FILTER (WHERE d.hhid IS NULL) AS unmatched_demographic_years
    FROM observed_hh_years o
    LEFT JOIN demo_keys d USING (hhid, panel_year)
    GROUP BY o.panel_year ORDER BY o.panel_year""").df()
display(coverage)
coverage.to_csv(AUDIT / 'annual_demographic_coverage.csv', index=False)

# Use the strict matched intersection. This excludes households removed in Step 02
# and household-years for which annual demographic extraction is unavailable.
# Cast the seven events to nullable integer, controls to numeric.
fields = ["k.hhid", "k.panel_year"]
fields += [f"TRY_CAST(NULLIF(TRIM(d.{name(x)}), '') AS INTEGER) AS {name(x)}" for x in EVENTS]
fields += ["TRY_CAST(NULLIF(TRIM(d.K_1), '') AS INTEGER) AS K_1",
           "TRY_CAST(NULLIF(TRIM(d.income10k), '') AS DOUBLE) AS income10k",
           "TRY_CAST(NULLIF(TRIM(d.family_size_numeric), '') AS DOUBLE) AS family_size_numeric"]
if "valid_transition" in optional:
    fields.append("TRY_CAST(NULLIF(TRIM(d.valid_transition), '') AS BOOLEAN) AS valid_transition")

con.execute(f"""COPY (
    SELECT {', '.join(fields)}
    FROM observed_hh_years k INNER JOIN demo_raw d
      ON k.hhid=TRIM(d.hhid) AND k.panel_year=TRY_CAST(d.panel_year AS INTEGER)
) TO {lit(ANNUAL_OUT)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
annual = pq(ANNUAL_OUT)
annual_check = con.execute(f"""SELECT COUNT(*) household_years, COUNT(DISTINCT hhid) households,
    COUNT(*)-COUNT(DISTINCT (hhid,panel_year)) duplicate_keys
    FROM {annual}""").df()
display(annual_check)
assert annual_check.duplicate_keys.iloc[0] == 0
print('Annual covariates:', ANNUAL_OUT)


,panel_year,purchase_household_years,matched_demographic_years,unmatched_demographic_years
0,2008,42862,41135,1727
1,2009,47965,45553,2412
2,2010,55268,47612,7656
3,2011,51661,44869,6792
4,2012,52650,46053,6597
5,2013,53660,46932,6728
6,2014,54300,47829,6471
7,2015,54776,48622,6154
8,2016,49935,43503,6432
9,2017,45333,39222,6111


,household_years,households,duplicate_keys
0,451330,80414,0


Annual covariates: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\06_hmm_master\hmm_annual_covariates.parquet


## C — Join annual eligibility with trip emissions and trip food spending

Trip source: six-category corrected Parquet (`0` indicates nonpurchase). Food source: Step 02 trip-level amounts. Each is independently required to have **unique four-column trip keys**. The join is left-sided from the emission universe and filtered against the matched annual index; food spending remains missing if the Step 02 row is missing or did not match the original purchases.

We do **not** replace missing `food_spend` with 0, and do **not** construct `healthy_food_spend` by applying `category_y` to category spending.


In [12]:
# Cell 4 — Annual partition loop: validation, join and audit
EXPECTED = {'cereal_y': (0,1,2,3), **{y:(0,1,2) for y in OUTCOMES if y!='cereal_y'}}
rows = []
for year in YEARS:
    started = time.perf_counter()
    cfile = CATEGORY_PANEL / f'panel_year={year}' / 'trips.parquet'
    ffile = FOOD_PANEL / f'panel_year={year}' / 'trips.parquet'
    c, f = pq(cfile), pq(ffile)
    ccols = set(con.execute(f'DESCRIBE SELECT * FROM {c}').df().column_name)
    fcols = set(con.execute(f'DESCRIBE SELECT * FROM {f}').df().column_name)
    assert set(KEYS + OUTCOMES + SPENDS) <= ccols, (year, 'category fields', sorted(set(KEYS+OUTCOMES+SPENDS)-ccols))
    assert set(KEYS + FOOD_FIELDS) <= fcols, (year, 'food fields', sorted(set(KEYS+FOOD_FIELDS)-fcols))

    # Trip identifiers remain strings; DATE cast only for date comparison.
    def view_sql(rel):
        return f"""SELECT CAST(hhid AS VARCHAR) hhid, CAST(panel_year AS INTEGER) panel_year,
                  CAST(trip_id AS VARCHAR) trip_id, TRY_CAST(purchase_date AS DATE) purchase_date,
                  * EXCLUDE (hhid, panel_year, trip_id, purchase_date)
                  FROM {rel}"""
    con.execute(f'CREATE OR REPLACE TEMP VIEW cat_y AS {view_sql(c)}')
    con.execute(f'CREATE OR REPLACE TEMP VIEW food_y AS {view_sql(f)}')
    checks = []
    for view in ('cat_y','food_y'):
        count, bad, duplicate = con.execute(f"""SELECT COUNT(*),
          COUNT(*) FILTER (WHERE hhid IS NULL OR TRIM(hhid)='' OR trip_id IS NULL OR TRIM(trip_id)=''
              OR purchase_date IS NULL OR panel_year <> {year} OR YEAR(purchase_date) <> {year}),
          COUNT(*)-COUNT(DISTINCT (hhid,panel_year,trip_id,purchase_date))
          FROM {view}""").fetchone()
        assert bad == 0 and duplicate == 0, (year,view,'invalid key',bad,'duplicate',duplicate)
        checks.append(count)
    bad_y = con.execute('SELECT ' + ','.join(
        f'COUNT(*) FILTER (WHERE {name(y)} IS NULL OR {name(y)} NOT IN ({",".join(map(str,EXPECTED[y]))}))'
        for y in OUTCOMES) + ' FROM cat_y').fetchone()
    assert not any(bad_y), (year, dict(zip(OUTCOMES,bad_y)))
    # The food table must not multiply category trips; missing food is preserved.
    target = TRIP_OUT / f'panel_year={year}' / 'trips.parquet'
    target.parent.mkdir(parents=True, exist_ok=True)
    selection = ', '.join('c.' + name(x) for x in OUTCOMES + SPENDS)
    con.execute(f"""COPY (
      SELECT c.hhid, c.panel_year, c.trip_id, c.purchase_date,
             {selection},
             f.regular_food_spend, f.magnet_food_spend, f.food_spend,
             f.food_trip_matched
      FROM cat_y c
      INNER JOIN {annual} a ON c.hhid=a.hhid AND c.panel_year=a.panel_year
      LEFT JOIN food_y f ON c.hhid=f.hhid AND c.panel_year=f.panel_year
        AND c.trip_id=f.trip_id AND c.purchase_date=f.purchase_date
    ) TO {lit(target)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    o = pq(target)
    metrics = con.execute(f"""SELECT COUNT(*) AS output_trips,
      COUNT(*)-COUNT(DISTINCT (hhid,panel_year,trip_id,purchase_date)) AS duplicate_output_keys,
      COUNT(*) FILTER (WHERE food_trip_matched IS TRUE) AS food_source_matched,
      COUNT(*) FILTER (WHERE food_trip_matched IS DISTINCT FROM TRUE) AS food_source_unmatched,
      COUNT(*) FILTER (WHERE food_spend IS NULL) AS null_food_spend,
      COUNT(*) FILTER (WHERE food_spend < 0 OR regular_food_spend < 0 OR magnet_food_spend < 0) AS negative_food_spend,
      COUNT(*) FILTER (WHERE food_trip_matched IS TRUE AND
          ABS(food_spend - (regular_food_spend + magnet_food_spend)) > 0.0001) AS inconsistent_food_totals
      FROM {o}""").fetchone()
    keys = ['output_trips','duplicate_output_keys','food_source_matched','food_source_unmatched',
            'null_food_spend','negative_food_spend','inconsistent_food_totals']
    r = dict(zip(keys,metrics))
    assert r['duplicate_output_keys']==0 and r['negative_food_spend']==0 and r['inconsistent_food_totals']==0, (year,r)
    r.update(panel_year=year,category_input_trips=checks[0],food_input_trips=checks[1],
             excluded_for_no_annual=checks[0]-r['output_trips'])
    r['food_match_rate'] = r['food_source_matched']/r['output_trips'] if r['output_trips'] else None
    r['seconds'] = round(time.perf_counter()-started,1)
    rows.append(r)
    print(f'{year}: {r["output_trips"]:,} trips, food match {r["food_match_rate"]:.1%}')
    if r['food_match_rate'] < MIN_FOOD_MATCH_RATE:
        print(f'  WARNING: food matching below {MIN_FOOD_MATCH_RATE:.0%}. Review Step 02 audit.')
    pd.DataFrame(rows).to_csv(AUDIT/'hmm_master_trip_merge_audit.csv',index=False)
display(pd.DataFrame(rows))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2008: 3,275,726 trips, food match 100.0%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2009: 3,567,676 trips, food match 100.0%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2010: 3,564,508 trips, food match 100.0%


... (11 outputs left out)

2016: 3,733,987 trips, food match 100.0%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2017: 3,351,241 trips, food match 100.0%


,output_trips,duplicate_output_keys,food_source_matched,food_source_unmatched,null_food_spend,negative_food_spend,inconsistent_food_totals,panel_year,category_input_trips,food_input_trips,excluded_for_no_annual,food_match_rate,seconds
0,3275726,0,3275726,0,0,0,0,2008,3368393,3368393,92667,1.0,3.4
1,3567676,0,3567676,0,0,0,0,2009,3699872,3699872,132196,1.0,3.9
2,3564508,0,3564508,0,0,0,0,2010,4027911,4027911,463403,1.0,4.2
3,3786058,0,3786058,0,0,0,0,2011,4206761,4206761,420703,1.0,4.6
4,3741219,0,3741219,0,0,0,0,2012,4154835,4154835,413616,1.0,4.6
5,3712444,0,3712444,0,0,0,0,2013,4119659,4119659,407215,1.0,4.5
6,3799699,0,3799699,0,0,0,0,2014,4196848,4196848,397149,1.0,4.6
7,3784228,0,3784228,0,0,0,0,2015,4145029,4145029,360801,1.0,4.5
8,3733987,0,3733987,0,0,0,0,2016,4122803,4122803,388816,1.0,4.5
9,3351241,0,3351241,0,0,0,0,2017,3704316,3704316,353075,1.0,3.8


## D — Final cross-layer checks and estimation readiness

This checks that every output trip has exactly one annual household-year, and that no annual household-year was accidentally duplicated by the trip join. Missing event covariates are **not** zero-imputed. Check `moving` missingness before passing covariates to a model whose transition design requires complete cases.


In [13]:
# Cell 5 — Final checks and concise model-preparation audit
hmm_paths = [TRIP_OUT/f'panel_year={year}'/'trips.parquet' for year in YEARS]
hmm_rel = 'read_parquet([' + ','.join(lit(p) for p in hmm_paths) + '])'
summary = con.execute(f"""WITH
    t AS (SELECT COUNT(*) trips, COUNT(DISTINCT hhid) households,
          COUNT(DISTINCT (hhid,panel_year)) trip_household_years
          FROM {hmm_rel}),
    a AS (SELECT COUNT(*) annual_rows, COUNT(DISTINCT hhid) annual_households FROM {annual})
    SELECT * FROM t CROSS JOIN a""").df()
display(summary)
assert summary.trip_household_years.iloc[0] == summary.annual_rows.iloc[0], 'Annual/trip household-year mismatch'
missing = con.execute('SELECT ' + ','.join(
    f'COUNT(*) FILTER (WHERE {name(x)} IS NULL) AS {name(x+"_missing")}'
    for x in EVENTS+CONTROLS) + f' FROM {annual}').df()
display(missing.T.rename(columns={0:'missing_household_years'}))
summary.to_csv(AUDIT/'hmm_master_overall_summary.csv', index=False)
missing.to_csv(AUDIT/'hmm_master_annual_missingness.csv', index=False)
print('READY FOR REVIEW — trip/annual indexing validated. Model estimation should use')
print('  Annual :', ANNUAL_OUT)
print('  Trips  :', TRIP_OUT)
print('Review the food match rates and missing annual transition covariates before fitting the HMM.')


,trips,households,trip_household_years,annual_rows,annual_households
0,36316786,80414,451330,451330,80414


,missing_household_years
event_new_child_missing,25187
getting_married_missing,25187
getting_divorce_or_separation_missing,25187
moving_missing,25187
getting_job_missing,25187
retirement_missing,25187
other_unemployment_missing,25187
K_1_missing,0
income10k_missing,55087
family_size_numeric_missing,0


READY FOR REVIEW — trip/annual indexing validated. Model estimation should use
  Annual : C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\06_hmm_master\hmm_annual_covariates.parquet
  Trips  : C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\06_hmm_master\hmm_trips
Review the food match rates and missing annual transition covariates before fitting the HMM.



# V38 Step 07 — Impute Annual Household Income

This section reads the annual covariates created in Step 06 and writes a distinct income-imputed annual file. Use that file for HMM estimation. The trip emission partitions remain unchanged.


In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_ROOT = Path("C:/Users/rwen13/Documents/healthy_purchase_data")
MASTER_ROOT = DATA_ROOT / "intermediate" / "v38_milk_beverage_combined" / "06_hmm_master"
INPUT_FILE = MASTER_ROOT / "hmm_annual_covariates.parquet"
OUTPUT_FILE = MASTER_ROOT / "hmm_annual_covariates_income_imputed.parquet"
AUDIT_DIR = MASTER_ROOT / "validation"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Run Step 03 first: {INPUT_FILE}")

annual = pd.read_parquet(INPUT_FILE)
required = {"hhid", "panel_year", "income10k"}
missing_columns = required - set(annual.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

# Household IDs are identifiers and must never be cast to numeric.
annual["hhid"] = annual["hhid"].astype("string")
annual["panel_year"] = pd.to_numeric(annual["panel_year"], errors="raise").astype("int16")
annual["income10k"] = pd.to_numeric(annual["income10k"], errors="coerce")
if annual[["hhid", "panel_year"]].isna().any().any():
    raise ValueError("Null household/year key in annual covariates")
if annual.duplicated(["hhid", "panel_year"]).any():
    raise ValueError("Expected one annual row per (hhid, panel_year)")
if annual["income10k"].notna().sum() == 0:
    raise ValueError("No observed income values available for imputation")

print(f"Input household-years: {len(annual):,}")
print(f"Input unique households: {annual['hhid'].nunique():,}")
print(f"Missing original income: {annual['income10k'].isna().sum():,}")

Input household-years: 451,330
Input unique households: 80,414
Missing original income: 55,087


In [15]:
# Load aligned household employment information from the demographic panel.
# The Step 03 compact annual file generally does not contain household employment status.
# IMPORTANT: set UNEMPLOYED_COLUMN to an actual binary variable in the demographic TSV:
# 1 = unemployed household, 0 = not unemployed, NA = unknown.
# Do NOT use an annual transition event flag in place of a current employment state.
DEMOGRAPHICS_FILE = DATA_ROOT / "processed/demographics/household_demographics_with_events_2008_2017.tsv"
# Current household unemployment status, not a life-event indicator.
UNEMPLOYED_COLUMN = "household_other_unemployed"

if not DEMOGRAPHICS_FILE.is_file():
    raise FileNotFoundError(f"Missing demographics: {DEMOGRAPHICS_FILE}")

# Inspect header without loading the large file if status column is not configured.
available_columns = pd.read_csv(DEMOGRAPHICS_FILE, sep="\t", nrows=0).columns.tolist()
if UNEMPLOYED_COLUMN is None or UNEMPLOYED_COLUMN not in available_columns:
    potential = [c for c in available_columns if any(t in c.lower() for t in ("employ", "work", "retir", "job", "labor"))]
    raise ValueError(
        "Set UNEMPLOYED_COLUMN to a VERIFIED current household unemployment flag (0/1) "
        "in the demographic source; do not use life-event flags. "
        f"Candidate columns: {potential}"
    )

employment = pd.read_csv(
    DEMOGRAPHICS_FILE, sep="\t", usecols=["hhid", "panel_year", UNEMPLOYED_COLUMN],
    dtype={"hhid": "string"}, low_memory=False
)
employment["panel_year"] = pd.to_numeric(employment["panel_year"], errors="raise").astype("int16")
employment[UNEMPLOYED_COLUMN] = pd.to_numeric(employment[UNEMPLOYED_COLUMN], errors="coerce")
if not employment[UNEMPLOYED_COLUMN].dropna().isin([0, 1]).all():
    raise ValueError("Unemployment flag must contain only 0, 1, or missing values")
if employment.duplicated(["hhid", "panel_year"]).any():
    raise ValueError("Duplicate household-year keys in aligned employment file")

annual = annual.merge(
    employment.rename(columns={UNEMPLOYED_COLUMN: "_household_unemployed"}),
    on=["hhid", "panel_year"], how="left", validate="one_to_one", sort=False
)
print("Unemployment status available:", annual["_household_unemployed"].notna().sum())
print("Missing income AND unemployed:",
      (annual["income10k"].isna() & annual["_household_unemployed"].eq(1)).sum())


Unemployment status available: 451330
Missing income AND unemployed: 5181


In [16]:
# Sort only for within-household time interpolation; restore source row order later.
work = annual.copy()
work["_original_row"] = np.arange(len(work), dtype=np.int64)
work = work.sort_values(["hhid", "panel_year"], kind="stable").reset_index(drop=True)
original_missing = work["income10k"].isna()
work["income_missing_indicator"] = original_missing.astype("int8")

# Batch interpolation from nearest OBSERVED income on each side within each household.
# Use calendar-year distances, including gaps; imposed unemployment zeros are not anchors.
unemployed_missing = original_missing & work["_household_unemployed"].eq(1).fillna(False)
anchors = pd.DataFrame({
    "value": work["income10k"].astype(float),
    "year": work["panel_year"].where(~original_missing).astype(float),
})
groups = anchors.groupby(work["hhid"], sort=False)
left = groups.ffill()
right = groups.bfill()
both = original_missing & left["value"].notna() & right["value"].notna()
filled = left["value"].combine_first(right["value"])
# Match np.interp's slope-first arithmetic for interior gaps.
slope = ((right.loc[both, "value"] - left.loc[both, "value"]) /
         (right.loc[both, "year"] - left.loc[both, "year"]))
filled.loc[both] = (left.loc[both, "value"] + slope *
                   (work.loc[both, "panel_year"] - left.loc[both, "year"]))
work["income10k_imputed"] = filled
# Release the intermediate arrays before the later household-year audits.
del anchors, groups, left, right, both, filled, slope

# Explicit override for missing-income, currently unemployed household-years.
work.loc[unemployed_missing, "income10k_imputed"] = 0.0

# Mark interior gaps separately from leading/trailing gaps.
observed_year = work["panel_year"].where(work["income10k"].notna())
first_obs = observed_year.groupby(work["hhid"]).transform("min")
last_obs = observed_year.groupby(work["hhid"]).transform("max")
work["income_imputation_method"] = "observed"
filled_from_household = original_missing & work["income10k_imputed"].notna()
work.loc[filled_from_household & (work["panel_year"] < first_obs), "income_imputation_method"] = "carry_backward"
work.loc[filled_from_household & (work["panel_year"] > last_obs), "income_imputation_method"] = "carry_forward"
work.loc[filled_from_household & work["panel_year"].between(first_obs, last_obs), "income_imputation_method"] = "linear_interpolation"

work.loc[unemployed_missing, "income_imputation_method"] = "unemployed_zero"

# Families without observed income in ANY available year: yearly observed median.
median_by_year = work.groupby("panel_year")["income10k"].median()
overall_median = work["income10k"].median()
fully_missing = work["income10k_imputed"].isna()
work.loc[fully_missing, "income10k_imputed"] = (
    work.loc[fully_missing, "panel_year"].map(median_by_year)
)
work.loc[fully_missing, "income_imputation_method"] = "year_median"
still_missing = work["income10k_imputed"].isna()
work.loc[still_missing, "income10k_imputed"] = overall_median
work.loc[still_missing, "income_imputation_method"] = "overall_median"

# Observed income always stays identical in both original and completed series.
assert np.allclose(
    work.loc[~original_missing, "income10k"],
    work.loc[~original_missing, "income10k_imputed"],
    equal_nan=True,
)
assert work["income10k_imputed"].notna().all()
assert work.loc[unemployed_missing, "income10k_imputed"].eq(0).all()
assert work.loc[unemployed_missing, "income_imputation_method"].eq("unemployed_zero").all()
assert (work["income_missing_indicator"] == work["income10k"].isna().astype("int8")).all()

# Keep the annual rows and all original columns in the original order.
result = (work.sort_values("_original_row", kind="stable")
              .drop(columns=["_original_row", "_household_unemployed"])
              .reset_index(drop=True))
assert len(result) == len(annual)
assert result[["hhid", "panel_year"]].equals(annual[["hhid", "panel_year"]].reset_index(drop=True))
print(result["income_imputation_method"].value_counts().to_string())

income_imputation_method
observed                396243
carry_forward            40385
unemployed_zero           5181
linear_interpolation      4421
year_median               2687
carry_backward            2413


In [17]:
# Save a NEW annual file rather than overwriting the Step 03 source.
result.to_parquet(OUTPUT_FILE, index=False)

by_year = (result.groupby("panel_year", as_index=False)
    .agg(annual_rows=("hhid", "size"),
         observed_income=("income_missing_indicator", lambda v: int((v == 0).sum())),
         missing_original=("income_missing_indicator", "sum"),
         missing_after=("income10k_imputed", lambda v: int(v.isna().sum()))))
by_year["original_missing_rate"] = by_year["missing_original"] / by_year["annual_rows"]
by_year.to_csv(AUDIT_DIR / "income_imputation_by_year.csv", index=False)

by_method = (result["income_imputation_method"].value_counts(dropna=False)
    .rename_axis("method").reset_index(name="household_years"))
by_method.to_csv(AUDIT_DIR / "income_imputation_by_method.csv", index=False)

by_household = (result.groupby("hhid")["income_missing_indicator"]
    .agg(missing_years="sum", total_years="size"))
by_household["observed_years"] = by_household["total_years"] - by_household["missing_years"]
summary = pd.DataFrame([{
    "households": len(by_household),
    "households_any_missing": int((by_household.missing_years > 0).sum()),
    "households_all_missing": int((by_household.observed_years == 0).sum()),
    "households_partially_missing": int(((by_household.missing_years > 0) & (by_household.observed_years > 0)).sum()),
    "original_missing_household_years": int(result.income_missing_indicator.sum()),
    "remaining_missing_household_years": int(result.income10k_imputed.isna().sum()),
}])
summary.to_csv(AUDIT_DIR / "income_imputation_summary.csv", index=False)

print("Saved:", OUTPUT_FILE)
print("\nHousehold summary:")
print(summary.to_string(index=False))
print("\nBy year:")
print(by_year.to_string(index=False))
print("\nBy method:")
print(by_method.to_string(index=False))

Saved: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\06_hmm_master\hmm_annual_covariates_income_imputed.parquet

Household summary:
 households  households_any_missing  households_all_missing  households_partially_missing  original_missing_household_years  remaining_missing_household_years
      80414                   49958                    2214                         47744                             55087                                  0

By year:
 panel_year  annual_rows  observed_income  missing_original  missing_after  original_missing_rate
       2008        41135            39014              2121              0               0.051562
       2009        45553            38295              7258              0               0.159331
       2010        47612            41363              6249              0               0.131248
       2011        44869            39131              5738              0               0.127883
       2

## Modeling note

Use `income10k_imputed` with the original `income_missing_indicator` as a possible control. The additional method category `unemployed_zero` identifies values imposed by the unemployment rule. Neither `other_unemployment` (an event) nor `retirement` (an event) establishes *current unemployment* by itself.

For temporal forecasting holdouts, refit imputation using training-period information only; do not leak future income into the training sample.

This notebook writes a NEW annual file and leaves all V38 Step 06 trip files unchanged. Point subsequent model loaders to `hmm_annual_covariates_income_imputed.parquet` explicitly.



# 2007 Initial Fresh-FV Share — Common HMM Starting Value

This section runs after Step 07 because it uses the household sample from the V38 HMM annual covariates. It writes the single pooled 2007 fresh-FV dollar share into the V38 `06_hmm_master` output directory.


# Step 04 — Common Initial Fresh-FV Share (2007)

Calculate **one pooled 2007 fresh-FV dollar share** for households in the current HMM sample:

\[
\pi_{\mathrm{FV},2007}=\frac{\sum_i\mathrm{FreshFVSpend}_{i,2007}}{\sum_i\mathrm{FoodSpend}_{i,2007}}.
\]

This is **not** the unweighted average of household shares. It supplies a common numerical **starting value** for the initial HMM healthy-state probability, not a fixed estimated parameter.

The 2007 produce and Magnet extraction TSVs are already prepared. All identifiers remain strings; regular and Magnet spending are reconciled before summation. Output is a **single-row Parquet with one column** (`initial_fresh_fv_share_2007`).


In [18]:
from pathlib import Path
import pandas as pd
import duckdb

DATA_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
REPO_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_cycle")
HMS_ROOT = Path(r"C:/Users/rwen13/Documents/2004-2020_Panel_Data/nielsen_extracts/HMS")

PRODUCE_FILE = DATA_ROOT / "intermediate/produce_purchases/fv_purchases_2007.tsv"
MAGNET_FILE = DATA_ROOT / "intermediate/magnet_food_purchases/magnet_food_purchases_2007.tsv"
PURCHASES_FILE = HMS_ROOT / "2007/Annual_Files/purchases_2007.tsv"
TRIPS_FILE = HMS_ROOT / "2007/Annual_Files/trips_2007.tsv"
PRODUCTS_FILE = HMS_ROOT / "Master_Files/Latest/products.tsv"
FOOD_LIST_FILE = REPO_ROOT / "data_cleaning/upc_classification/food_list.xlsx"
ANNUAL_FILE = DATA_ROOT / "intermediate/v38_milk_beverage_combined/06_hmm_master/hmm_annual_covariates_income_imputed.parquet"
OUTPUT_DIR = DATA_ROOT / "intermediate/v38_milk_beverage_combined/06_hmm_master"
AUDIT_DIR = OUTPUT_DIR / "validation"
OUTPUT_FILE = OUTPUT_DIR / "hmm_initial_fresh_fv_share_2007.parquet"

for name, path in {
    "Produce": PRODUCE_FILE, "Magnet": MAGNET_FILE,
    "Raw purchases": PURCHASES_FILE, "Raw trips": TRIPS_FILE,
    "Product master": PRODUCTS_FILE, "Food module list": FOOD_LIST_FILE,
    "Current HMM annual panel": ANNUAL_FILE,
}.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {name}: {path}")
    print(f"Found {name}: {path}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "temp").mkdir(parents=True, exist_ok=True)

def sql_path(path):
    return "'" + path.as_posix().replace("'", "''") + "'"

def source_tsv(path):
    # All columns begin as text; household/trip/UPC join keys cannot lose zeros.
    return f"read_csv({sql_path(path)}, delim='\t', header=true, all_varchar=true)"

if 'con' in globals():
    con.close()
INITIAL_TEMP = DATA_ROOT / "temp/v38_milk_beverage_combined/initial_2007"
INITIAL_TEMP.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(database=":memory:")
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")
con.execute(f"SET temp_directory={sql_path(INITIAL_TEMP)}")


Found Produce: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\produce_purchases\fv_purchases_2007.tsv
Found Magnet: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\magnet_food_purchases\magnet_food_purchases_2007.tsv
Found Raw purchases: C:\Users\rwen13\Documents\2004-2020_Panel_Data\nielsen_extracts\HMS\2007\Annual_Files\purchases_2007.tsv
Found Raw trips: C:\Users\rwen13\Documents\2004-2020_Panel_Data\nielsen_extracts\HMS\2007\Annual_Files\trips_2007.tsv
Found Product master: C:\Users\rwen13\Documents\2004-2020_Panel_Data\nielsen_extracts\HMS\Master_Files\Latest\products.tsv
Found Food module list: C:\Users\rwen13\Documents\healthy_purchase_cycle\data_cleaning\upc_classification\food_list.xlsx
Found Current HMM annual panel: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\06_hmm_master\hmm_annual_covariates_income_imputed.parquet


In [19]:
# The food-module lookup is a small permitted rule file, not Nielsen purchase data.
# Normalize MODULE codes only; never normalize household, UPC, or trip identifiers.
food = pd.read_excel(FOOD_LIST_FILE, dtype={"product_module_code": "string"})
if "product_module_code" not in food.columns:
    raise ValueError("food_list.xlsx must contain product_module_code")
food_modules = (
    food["product_module_code"].dropna().str.strip()
    .str.replace(r"\.0+$", "", regex=True)
    .str.lstrip("0").replace("", "0").drop_duplicates()
)
if food_modules.empty:
    raise ValueError("The food-module list is empty")
con.register("food_module_frame", pd.DataFrame({"module_code": food_modules.astype(str)}))
con.execute("CREATE OR REPLACE TEMP TABLE food_modules AS SELECT DISTINCT module_code FROM food_module_frame")

# Restrict processing to households retained in the HMM model sample.
con.execute(f"""
CREATE OR REPLACE TEMP TABLE hmm_households AS
SELECT DISTINCT CAST(hhid AS VARCHAR) AS hhid
FROM read_parquet({sql_path(ANNUAL_FILE)})
WHERE hhid IS NOT NULL
""")
print("HMM households:", con.execute("SELECT COUNT(*) FROM hmm_households").fetchone()[0])


HMM households: 80414


In [20]:
# Read extracted regular produce and Magnet food from 2007.
# Use 2007 calendar purchase dates rather than trusting file-year labels.
con.execute(f"""
CREATE OR REPLACE TEMP TABLE produce_2007 AS
SELECT CAST(household_code AS VARCHAR) AS hhid,
       CAST(trip_code_uc AS VARCHAR) AS trip_code_uc,
       CAST(upc AS VARCHAR) AS upc,
       CAST(upc_ver_uc AS VARCHAR) AS upc_ver_uc,
       TRY_CAST(final_price_paid AS DOUBLE) AS net_amount
FROM {source_tsv(PRODUCE_FILE)}
WHERE YEAR(TRY_CAST(purchase_date AS DATE)) = 2007
  AND TRY_CAST(fv_class AS INTEGER) = 1
""")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE magnet_2007 AS
SELECT CAST(household_code AS VARCHAR) AS hhid,
       CAST(trip_code_uc AS VARCHAR) AS trip_code_uc,
       CAST(upc AS VARCHAR) AS upc,
       CAST(upc_ver_uc AS VARCHAR) AS upc_ver_uc,
       TRY_CAST(magnet_food_spend_increment_capped AS DOUBLE) AS food_amount,
       TRY_CAST(magnet_fresh_fv_spend_increment_capped AS DOUBLE) AS fresh_amount
FROM {source_tsv(MAGNET_FILE)}
WHERE YEAR(TRY_CAST(purchase_date AS DATE)) = 2007
  AND TRY_CAST(is_magnet_food AS INTEGER) = 1
""")

# Preserve repeated purchase lines; create a key-level membership set only for
# anti-joining raw Regular purchases, not for deleting or summing Magnet lines.
con.execute("""
CREATE OR REPLACE TEMP TABLE magnet_keys AS
SELECT trip_code_uc, upc, upc_ver_uc, COUNT(*) AS magnet_line_count
FROM magnet_2007 GROUP BY 1,2,3
""")

# Produce and Magnet should not double-count the same purchase UPC/trip.
produce_magnet_overlap = con.execute("""
SELECT COUNT(*) FROM produce_2007 p
JOIN magnet_keys m USING (trip_code_uc, upc, upc_ver_uc)
""").fetchone()[0]
if produce_magnet_overlap:
    raise ValueError(f"Produce/Magnet overlap: {produce_magnet_overlap} lines. Reconcile before calculating shares.")
print("Regular Fresh FV vs Magnet overlapping lines:", produce_magnet_overlap)


Regular Fresh FV vs Magnet overlapping lines: 0


In [21]:
# Source purchases are read only for the 2007 source file.
# Join raw purchases to actual trip dates and the product master using string keys.
# The product reference must have a unique (upc, upc_ver_uc) key.
con.execute(f"""
CREATE OR REPLACE TEMP TABLE raw_trips AS
SELECT CAST(trip_code_uc AS VARCHAR) AS trip_code_uc,
       CAST(household_code AS VARCHAR) AS hhid,
       TRY_CAST(purchase_date AS DATE) AS purchase_date
FROM {source_tsv(TRIPS_FILE)}
""")
con.execute(f"""
CREATE OR REPLACE TEMP TABLE product_modules AS
SELECT CAST(upc AS VARCHAR) AS upc,
       CAST(upc_ver_uc AS VARCHAR) AS upc_ver_uc,
       COALESCE(NULLIF(LTRIM(REGEXP_REPLACE(TRIM(product_module_code), '\.0+$', ''), '0'), ''), '0') AS module_code
FROM {source_tsv(PRODUCTS_FILE)}
""")
product_duplicates = con.execute("""
SELECT COUNT(*) FROM (
  SELECT upc, upc_ver_uc FROM product_modules
  GROUP BY 1,2 HAVING COUNT(*) > 1
)
""").fetchone()[0]
if product_duplicates:
    raise ValueError(f"Duplicate UPC/version pairs in product master: {product_duplicates}")
trip_duplicates = con.execute("""
SELECT COUNT(*) FROM (
  SELECT trip_code_uc FROM raw_trips GROUP BY 1 HAVING COUNT(*) > 1
)
""").fetchone()[0]
if trip_duplicates:
    raise ValueError(f"Duplicate trip_code_uc in 2007 trips: {trip_duplicates}")

initial_purchase_cache = INITIAL_TEMP / '_raw_2007_projection.parquet'
print('Reading 2007 purchases once...', flush=True)
con.execute(f"""COPY (
SELECT t.hhid, p.trip_code_uc, p.upc, p.upc_ver_uc,
       TRY_CAST(p.total_price_paid AS DOUBLE) AS total_price_paid,
       TRY_CAST(p.coupon_value AS DOUBLE) AS coupon_value
FROM {source_tsv(PURCHASES_FILE)} p
JOIN raw_trips t ON p.trip_code_uc = t.trip_code_uc
WHERE YEAR(t.purchase_date) = 2007

) TO {sql_path(initial_purchase_cache)} (FORMAT PARQUET, COMPRESSION SNAPPY)""")
con.execute(f"CREATE OR REPLACE TEMP VIEW raw_2007 AS SELECT * FROM read_parquet({sql_path(initial_purchase_cache)})")

# Reconcile Magnet multiplicities against source purchases.
# Repeated UPCs within a trip are valid purchase lines, NOT duplicates to delete.
con.execute("""
CREATE OR REPLACE TEMP TABLE source_magnet_counts AS
SELECT r.trip_code_uc, r.upc, r.upc_ver_uc, COUNT(*) AS raw_line_count
FROM raw_2007 r
JOIN magnet_keys m USING (trip_code_uc, upc, upc_ver_uc)
GROUP BY 1,2,3
""")
key_mismatches = con.execute("""
SELECT COUNT(*) FROM magnet_keys m
LEFT JOIN source_magnet_counts r USING (trip_code_uc, upc, upc_ver_uc)
WHERE r.raw_line_count IS NULL OR r.raw_line_count <> m.magnet_line_count
""").fetchone()[0]
if key_mismatches:
    raise ValueError(
        f"Magnet/raw line multiplicities differ for {key_mismatches} UPC-trip keys. "
        "Check source-year overlap or purchase-date alignment; do not silently deduplicate."
    )
print("Magnet/raw UPC-trip count mismatches:", key_mismatches)


<>:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\rwen13\AppData\Local\Temp\ipykernel_4676\3251584585.py:21: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  GROUP BY 1,2 HAVING COUNT(*) > 1


Reading 2007 purchases once...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Magnet/raw UPC-trip count mismatches: 0


In [22]:
# Sum annual spending within the household. The Magnet overlap is removed
# from Regular food BEFORE aggregation. All dollars use max(price-coupon, 0).
# Missing prices are not treated as observed zeros; report their number below.
con.execute("""
CREATE OR REPLACE TEMP TABLE regular_food_by_hh AS
SELECT r.hhid,
       SUM(GREATEST(r.total_price_paid - COALESCE(r.coupon_value, 0), 0)) AS regular_food_spend,
       COUNT(*) AS regular_food_lines,
       COUNT(*) FILTER (WHERE r.total_price_paid IS NULL) AS regular_missing_price_lines
FROM raw_2007 r
JOIN product_modules p ON r.upc=p.upc AND r.upc_ver_uc=p.upc_ver_uc
JOIN food_modules f ON p.module_code=f.module_code
ANTI JOIN magnet_keys m ON r.trip_code_uc=m.trip_code_uc AND r.upc=m.upc AND r.upc_ver_uc=m.upc_ver_uc
GROUP BY 1
""")
con.execute("""
CREATE OR REPLACE TEMP TABLE regular_fresh_by_hh AS
SELECT hhid,
       SUM(GREATEST(net_amount, 0)) AS regular_fresh_fv_spend,
       COUNT(*) AS fresh_lines,
       COUNT(*) FILTER (WHERE net_amount IS NULL) AS fresh_missing_price_lines
FROM produce_2007 GROUP BY 1
""")
con.execute("""
CREATE OR REPLACE TEMP TABLE magnet_by_hh AS
SELECT hhid,
       SUM(food_amount) AS magnet_food_spend,
       SUM(fresh_amount) AS magnet_fresh_fv_spend,
       COUNT(*) AS magnet_lines,
       COUNT(*) FILTER (WHERE food_amount IS NULL OR fresh_amount IS NULL) AS magnet_missing_price_lines
FROM magnet_2007 GROUP BY 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE household_spending AS
SELECT h.hhid,
       COALESCE(r.regular_food_spend,0) AS regular_food_spending_2007,
       COALESCE(m.magnet_food_spend,0) AS magnet_food_spending_2007,
       COALESCE(f.regular_fresh_fv_spend,0) AS regular_fresh_fv_spending_2007,
       COALESCE(m.magnet_fresh_fv_spend,0) AS magnet_fresh_fv_spending_2007,
       COALESCE(r.regular_food_spend,0)+COALESCE(m.magnet_food_spend,0) AS total_food_spending_2007,
       COALESCE(f.regular_fresh_fv_spend,0)+COALESCE(m.magnet_fresh_fv_spend,0) AS fresh_fv_spending_2007,
       COALESCE(r.regular_food_lines,0) AS regular_food_lines,
       COALESCE(f.fresh_lines,0) AS regular_fresh_lines,
       COALESCE(m.magnet_lines,0) AS magnet_lines,
       COALESCE(r.regular_missing_price_lines,0) + COALESCE(f.fresh_missing_price_lines,0) + COALESCE(m.magnet_missing_price_lines,0) AS missing_price_lines
FROM hmm_households h
LEFT JOIN regular_food_by_hh r USING(hhid)
LEFT JOIN regular_fresh_by_hh f USING(hhid)
LEFT JOIN magnet_by_hh m USING(hhid)
""")

# No 2007 spending or a zero denominator means undefined share (NULL), not 0.
# Fresh FV is a subset of all food purchases; violations should block output.
invalid = con.execute("""
SELECT COUNT(*) FROM household_spending
WHERE fresh_fv_spending_2007 > total_food_spending_2007 + 0.01
   OR fresh_fv_spending_2007 < -0.01 OR total_food_spending_2007 < -0.01
""").fetchone()[0]
if invalid:
    raise ValueError(f"{invalid} households have fresh FV spending greater than food spending or negative totals")



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [23]:
# One pooled dollar share across all current HMM households (not a mean of household shares).
# All components below are already net of coupons, capped at zero, and adjusted
# for Regular/Magnet overlap in the preceding cells.
result = con.execute("""
SELECT
    SUM(fresh_fv_spending_2007) AS fresh_fv_dollars,
    SUM(total_food_spending_2007) AS food_dollars,
    COUNT(*) AS hmm_households,
    COUNT(*) FILTER (WHERE total_food_spending_2007 > 0) AS households_with_food_spend,
    SUM(missing_price_lines) AS missing_price_lines
FROM household_spending
""").fetchone()

fresh_fv_dollars, food_dollars, n_households, n_food_households, missing_price_lines = result
if food_dollars is None or food_dollars <= 0:
    raise ValueError("Cannot calculate common initial share: total 2007 food spending must be positive")
common_share = float(fresh_fv_dollars / food_dollars)
if not (0 <= common_share <= 1):
    raise ValueError(f"Initial share is outside [0, 1]: {common_share}")

# Write exactly ONE numeric value, with no household identifiers or household-level rows.
con.execute(f"""
COPY (
    SELECT CAST({common_share:.17g} AS DOUBLE) AS initial_fresh_fv_share_2007
) TO {sql_path(OUTPUT_FILE)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print(f"initial_fresh_fv_share_2007 = {common_share:.8f}")
print(f"HMM households: {n_households:,}; households with positive 2007 food spending: {n_food_households:,}")
print(f"2007 Fresh FV dollars: {fresh_fv_dollars:,.2f}; Food dollars: {food_dollars:,.2f}")
print(f"Purchase lines with missing prices (audit): {missing_price_lines:,}")
print(f"Saved single-value Parquet: {OUTPUT_FILE}")
con.close()

initial_purchase_cache.unlink()  # The connection is closed; the temporary projection is no longer needed.


initial_fresh_fv_share_2007 = 0.06155548
HMM households: 80,414; households with positive 2007 food spending: 35,348
2007 Fresh FV dollars: 5,500,406.64; Food dollars: 89,356,896.00
Purchase lines with missing prices (audit): 0
Saved single-value Parquet: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\06_hmm_master\hmm_initial_fresh_fv_share_2007.parquet


### Interpretation

- The file contains **one row and one numeric column**. It is not a household-level dataset.
- The pooled dollar share is calculated from total fresh-FV and food spending across the selected HMM household sample.
- The result may be used as a starting value for a common HMM initial-state probability; it is **not** itself a latent-state probability estimate.
- This notebook uses purchase lines from the **2007 source-year file**; source-year spillover must be checked separately if needed.
